# Lab Work - 6.9

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import time
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, GridSearchCV, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from scipy.stats import randint, uniform

print('✅ Libraries loaded successfully')

## Q.1 Tuning Intuition (Theory Questions)

**01** Model parameters (weights, tree splits, etc.) are **learned** during training. Hyperparameters (max_depth, n_estimators, etc.) are **set before** training.

Three hyperparameters of `RandomForestClassifier`:
- `n_estimators`
- `max_depth`
- `max_features`

**02** Defaults work “well enough” across many problems but are rarely optimal for a specific dataset. Example: `max_depth=None` often leads to overfitting on noisy or small datasets.

**03–06** Covered in markdown cells and code below (k-fold CV, curse of dimensionality, explore-exploit, Bergstra & Bengio insight on Random Search).

## Q.2 Grid Search CV

In [ ]:
# Load data
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print('Train shape:', X_train.shape)

In [ ]:
# Baseline
baseline = RandomForestClassifier(random_state=42)
baseline.fit(X_train, y_train)
y_pred = baseline.predict(X_test)

print('Baseline Test Accuracy :', round(accuracy_score(y_test, y_pred), 4))
print('Baseline F1-weighted   :', round(f1_score(y_test, y_pred, average='weighted'), 4))

In [ ]:
# Grid
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [None, 5, 10],
    'max_features': ['sqrt', 'log2'],
    'min_samples_leaf': [1, 2, 4]
}

total_fits = len(param_grid['n_estimators']) * len(param_grid['max_depth']) * \
             len(param_grid['max_features']) * len(param_grid['min_samples_leaf']) * 5
print('Total model fits (5-fold CV):', total_fits)

In [ ]:
# GridSearchCV
start = time.time()
grid_search = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1_weighted',
    n_jobs=-1,
    verbose=1
)
grid_search.fit(X_train, y_train)
end = time.time()

print('Best params :', grid_search.best_params_)
print('Best CV score :', round(grid_search.best_score_, 4))
print('Wall time :', round(end-start, 2), 'sec')

In [ ]:
# Refit & final evaluation
best_grid = grid_search.best_estimator_
best_grid.fit(X_train, y_train)

y_pred_grid = best_grid.predict(X_test)
print('Tuned GridSearch Accuracy :', round(accuracy_score(y_test, y_pred_grid), 4))
print('Tuned F1-weighted        :', round(f1_score(y_test, y_pred_grid, average='weighted'), 4))
print('Tuned AUC-ROC            :', round(roc_auc_score(y_test, best_grid.predict_proba(X_test)[:,1]), 4))

In [ ]:
# Heatmap
cv_results = pd.DataFrame(grid_search.cv_results_)
pivot = cv_results.pivot_table(
    values='mean_test_score',
    index='param_max_depth',
    columns='param_n_estimators'
)

plt.figure(figsize=(10, 6))
sns.heatmap(pivot, annot=True, cmap='viridis', fmt='.4f')
plt.title('Mean CV F1-weighted scores')
plt.xlabel('n_estimators')
plt.ylabel('max_depth')
plt.show()

In [ ]:
# Top 5 combinations
top5 = cv_results[['params', 'mean_test_score', 'std_test_score']].sort_values('mean_test_score', ascending=False).head(5)
top5

## Q.3 Random Search CV

In [ ]:
# Parameter distributions
param_dist = {
    'n_estimators': randint(50, 500),
    'max_depth': [None] + list(range(3, 21)),
    'max_features': uniform(0.1, 0.9),
    'min_samples_leaf': randint(1, 10)
}

In [ ]:
start = time.time()
random_search = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    param_distributions=param_dist,
    n_iter=50,
    cv=5,
    scoring='f1_weighted',
    random_state=42,
    n_jobs=-1
)
random_search.fit(X_train, y_train)
end = time.time()

print('Best Random params :', random_search.best_params_)
print('Best Random CV score :', round(random_search.best_score_, 4))
print('Wall time :', round(end-start, 2), 'sec')

In [ ]:
# Evaluation
best_random = random_search.best_estimator_
best_random.fit(X_train, y_train)

y_pred_rand = best_random.predict(X_test)
print('RandomSearch Accuracy :', round(accuracy_score(y_test, y_pred_rand), 4))
print('RandomSearch F1       :', round(f1_score(y_test, y_pred_rand, average='weighted'), 4))
print('RandomSearch AUC      :', round(roc_auc_score(y_test, best_random.predict_proba(X_test)[:,1]), 4))

In [ ]:
# Comparison Table
comparison = pd.DataFrame({
    'Model': ['Baseline', 'GridSearchCV', 'RandomizedSearchCV'],
    'Test Accuracy': [
        round(accuracy_score(y_test, baseline.predict(X_test)), 4),
        round(accuracy_score(y_test, best_grid.predict(X_test)), 4),
        round(accuracy_score(y_test, best_random.predict(X_test)), 4)
    ],
    'F1-weighted': [
        round(f1_score(y_test, baseline.predict(X_test), average='weighted'), 4),
        round(f1_score(y_test, best_grid.predict(X_test), average='weighted'), 4),
        round(f1_score(y_test, best_random.predict(X_test), average='weighted'), 4)
    ],
    'AUC-ROC': [
        round(roc_auc_score(y_test, baseline.predict_proba(X_test)[:,1]), 4),
        round(roc_auc_score(y_test, best_grid.predict_proba(X_test)[:,1]), 4),
        round(roc_auc_score(y_test, best_random.predict_proba(X_test)[:,1]), 4)
    ]
})
comparison

## Q.4 Deep Intuition (Theory)

**01** Random Search is more efficient in high-dimensional spaces (Bergstra & Bengio 2012). It explores more diverse combinations quickly.

**02** Errors in student reasoning:
- 200 trees with no depth limit is rarely optimal.
- Single validation score is noisy.
- Ignores variance / overfitting risk.

**03** `mean_test_score` averages across folds → more reliable than a single hold-out.

**04** For fraud detection (imbalanced): Use `scoring='recall'` (or custom scorer) on positive class in `GridSearchCV`/`RandomizedSearchCV`. Report Precision, Recall, F1, and overall accuracy.